# Laboratorio 11. Regularización y generalización

**Pregunta guía:** ¿Cómo pueden regularización y parada temprana reducir el sobreajuste?

## Objetivos
Al finalizar podrás comparar curvas train/validación; aplicar weight decay, dropout y early stopping; mantener test fuera de la selección.

## Prerrequisitos y conexión
Combina el ciclo de entrenamiento (05–08), activaciones (09) e inicialización (10). Se requiere Python básico; cada notebook puede ejecutarse de forma independiente desde un kernel limpio.

## Teoría breve
La regularización restringe soluciones: weight decay añade $\lambda\|W\|_2^2$ a la pérdida; dropout enmascara activaciones durante train; early stopping elige un checkpoint mediante validación. Ninguna técnica garantiza por sí sola menor error fuera de muestra.

## Problema y datos
Una red flexible ajusta una muestra pequeña y ruidosa de una función suave; comparamos decisiones de regularización. Todos los datos se generan en el notebook: no se requieren descargas ni archivos locales.

## Experimento conceptual
¿Qué diferencia entre pérdidas train y validación indica sobreajuste en este experimento? Fija la semilla, ejecuta las celdas en orden y cambia una decisión cada vez.

### Preparación del entorno
Importamos NumPy y PyTorch, fijamos semillas y elegimos CPU o GPU de forma opcional.


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch import nn
import torch.nn.functional as F

SEED = 2026
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.rcParams['figure.figsize'] = (7, 4)
print('PyTorch:', torch.__version__, '| dispositivo:', device)


**Resultado e interpretación.** La celda confirma el entorno utilizado. El resto del laboratorio está diseñado para CPU; una GPU disponible se utiliza sin ser requisito.


### Preparar particiones y modelos
Usamos datos sintéticos, con validación separada; el test no interviene en la selección.


In [ ]:
x_all = torch.linspace(-2, 2, 140, device=device).unsqueeze(1)
y_all = torch.sin(2.5 * x_all) + 0.25 * torch.randn_like(x_all)
train_x, val_x = x_all[:45], x_all[45:95]
train_y, val_y = y_all[:45], y_all[45:95]
configurations = {'sin_regularizar': (0.0, 0.0), 'weight_decay': (0.03, 0.0), 'dropout': (0.0, 0.25)}
curves = {}


**Resultado e interpretación.** Las particiones son contiguas por diseño del ejemplo; para datos temporales reales habría que definir el protocolo según el uso previsto.


### Entrenar y seleccionar
Mantenemos arquitectura y pasos; cambiamos weight decay o dropout y registramos pérdidas.


In [ ]:
for name, (decay, dropout_rate) in configurations.items():
    model = nn.Sequential(nn.Linear(1,64), nn.ReLU(), nn.Dropout(dropout_rate),
                          nn.Linear(64,64), nn.ReLU(), nn.Dropout(dropout_rate), nn.Linear(64,1)).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.01, weight_decay=decay)
    train_curve, val_curve = [], []
    for epoch in range(120):
        model.train(); train_loss = F.mse_loss(model(train_x), train_y)
        optimizer.zero_grad(); train_loss.backward(); optimizer.step()
        model.eval()
        with torch.no_grad(): val_loss = F.mse_loss(model(val_x), val_y).item()
        train_curve.append(train_loss.item()); val_curve.append(val_loss)
    curves[name] = (train_curve, val_curve)
    print(name, 'mínimo val:', round(min(val_curve), 4), '| época:', int(np.argmin(val_curve)))
for name, (tr, va) in curves.items(): plt.plot(va, label=name + ' val')
plt.xlabel('Época'); plt.ylabel('MSE validación'); plt.legend(); plt.show()


**Resultado e interpretación.** La menor pérdida de validación selecciona dentro de este conjunto de decisiones; se reserva un test independiente para una estimación final no sesgada por esa selección.


## Limitaciones y conclusiones clave
Los resultados dependen de esta semilla, tamaño de muestra, arquitectura y protocolo. No extrapoles métricas sintéticas a datos reales; separa decisiones de modelado de conclusiones generales. La celda de salida aporta la evidencia numérica y visual para responder la pregunta guía.

- Los parámetros y activaciones determinan cómo se representa la señal.
- La pérdida, el optimizador y el protocolo de evaluación forman parte del experimento.
- Una métrica aislada no describe todos los errores ni garantiza generalización.

## Ejercicios progresivos
1. **Guiado:** Guiado: añade checkpoint del mejor estado según validación y evalúa solo una vez en test.
2. **Independiente:** Independiente: cambia únicamente la tasa de dropout y compara curvas.
3. **Desafío:** Desafío: explica por qué aumentar regularización puede causar subajuste y no garantiza mejor desempeño.

## Conexión con el siguiente laboratorio
Usa la representación, operación o criterio de evaluación de este laboratorio como punto de partida del siguiente paso de la secuencia.
